# Sim2real on one Colab A100: UniCache fork, ALFRED

Replays the ALFRED subset on a real A100 with the UniCache vLLM fork, to check
whether the simulator's eviction-policy hit ratios hold on real hardware.

**Runtime:** set *Runtime → Change runtime type → A100 GPU* (High-RAM if offered).
Keep this tab open, because Colab disconnects idle sessions.

| Step | Time |
|---|---|
| 1. Check the GPU | 1 min |
| 2. Upload `sim2real-a100-bundle.tar.gz` (11 MB) | 1 min |
| 3. Build the fork (CUDA kernels for A100 only) | ~60–90 min, one time |
| 4. Checks: imports, policy sandbox | 1 min |
| 5. Run: smoke test, then ALFRED × {LRU, UniCache, LeCaR, Agent}, then LRU × 2 more | ~55 min |
| 6. Download `results-a100.tar.gz` | 1 min |

Nothing downloads model weights: the engine uses dummy weights and replays recorded outputs.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!nproc; free -g | head -2; python --version

## 2. Upload the bundle
Pick `sim2real-a100-bundle.tar.gz` from your laptop (`UniCache/result/sim2real-transfer/`). Alternatively, copy it to Google Drive and mount Drive.

In [ ]:
import os, tarfile
from google.colab import files
os.chdir('/content')
up = files.upload()                      # choose sim2real-a100-bundle.tar.gz
name = next(iter(up))
tarfile.open(name).extractall('/content')
print(os.listdir('/content/sim2real-a100-bundle'))

## 3. Build the fork (one time, ~60–90 min)
Installs torch 2.6 + xformers, then compiles the fork's CUDA kernels for the A100 only
(`TORCH_CUDA_ARCH_LIST=8.0`). The log goes to `/content/build.log`. The last lines are printed at the end.

In [ ]:
%%bash
set -e
cd /content/sim2real-a100-bundle/unicache
pip install -q torch==2.6.0 torchvision==0.21.0 --index-url https://download.pytorch.org/whl/cu124
sed -i '/^torchaudio/d;/^ray\[/d' requirements/cuda.txt     # not needed here
pip install -q -r requirements/build.txt "cmake<4"
export TORCH_CUDA_ARCH_LIST=8.0 VLLM_TARGET_DEVICE=cuda SETUPTOOLS_SCM_PRETEND_VERSION=0.1.dev22
export MAX_JOBS=$(( $(nproc) < 8 ? $(nproc) : 8 ))         # ~3 GB RAM per compile job
time pip install --no-build-isolation -e . > /content/build.log 2>&1 || { tail -40 /content/build.log; exit 1; }
tail -3 /content/build.log

## 4. Checks

In [ ]:
%%bash
cd /content/sim2real-a100-bundle/unicache
python -c "import torch, vllm, xformers; print('torch', torch.__version__, '| xformers', xformers.__version__, '|', torch.cuda.get_device_name(0))"
unshare --user --map-root-user --net true && echo "policy sandbox OK (unshare)"

## 5. Run (~55 min)
This stops at the first failure. Each run saves into `results-a100/` as it finishes.

In [ ]:
%%bash
cd /content/sim2real-a100-bundle
PY=python UNICACHE_POLICY_SANDBOX=unshare bash run_a100.sh 2>&1 | grep -vE "^(INFO|WARNING)" | tail -40

## 6. Download the results
Send `results-a100.tar.gz` back.

In [ ]:
from google.colab import files
files.download('/content/sim2real-a100-bundle/results-a100.tar.gz')